# ETAPA: Retomada R1

Este notebook executa a validacao R1: manifesto (manifest_unificado) vs arquivos reais no filesystem (dados_iniciais).

Regras:
- Somente leitura dos dados originais.
- Nao publicar/migrar para MinIO.
- Saidas serao gravadas em outputs/R1.


# ETAPA: Imports e configuracao


In [1]:
from __future__ import annotations

import os
import sys
import json
import time
import hashlib
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm import tqdm  # usar barra de texto (evita ipywidgets)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

print('Python:', sys.version)
print('Pandas:', pd.__version__)
print('Numpy:', np.__version__)



Python: 3.11.14 | packaged by conda-forge | (main, Oct 22 2025, 22:46:25) [GCC 14.3.0]
Pandas: 2.3.3
Numpy: 1.26.4


# ETAPA: Paths e utilitarios de descoberta


In [2]:
BASE_DIR = Path('/home/wilson/Maringa/fase_1_diagnostico')
NB_DIR = Path('/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/notebooks')
OUT_DIR = Path('/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R1')
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now().strftime('%Y-%m-%d_%H%M%S')


def find_unique_paths(base: Path, pattern: str) -> list[Path]:
    return sorted([p for p in base.rglob(pattern) if p.is_file()])


def find_unique_dirs(base: Path, name: str) -> list[Path]:
    return sorted([p for p in base.rglob(name) if p.is_dir()])


def choose_unique(candidates: list[Path], what: str) -> Path:
    if len(candidates) == 1:
        return candidates[0]
    print(f'ERRO: nao foi possivel determinar unicamente {what}.')
    print('Candidatos encontrados:')
    for i, c in enumerate(candidates, start=1):
        print(f'  {i:02d}: {c} (mtime={datetime.fromtimestamp(c.stat().st_mtime) if c.exists() else "NA"})')
    raise RuntimeError(f'Ambiguidade em {what}: ajuste a busca ou escolha manualmente.')


manifest_candidates = find_unique_paths(BASE_DIR, 'manifest_unificado.csv')
manifest_path = choose_unique(manifest_candidates, 'manifest_unificado.csv')

# diretorio dados_iniciais: tenta localizar por nome
raw_dir_candidates = find_unique_dirs(BASE_DIR, 'dados_iniciais')
dados_iniciais_dir = choose_unique(raw_dir_candidates, 'diretorio dados_iniciais')

print('BASE_DIR:', BASE_DIR)
print('manifest_path:', manifest_path)
print('dados_iniciais_dir:', dados_iniciais_dir)
print('OUT_DIR:', OUT_DIR)



BASE_DIR: /home/wilson/Maringa/fase_1_diagnostico
manifest_path: /home/wilson/Maringa/fase_1_diagnostico/dados/manifestos_final/manifest_unificado.csv
dados_iniciais_dir: /home/wilson/Maringa/fase_1_diagnostico/dados/dados_iniciais
OUT_DIR: /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R1


# ETAPA: Carregar manifesto


In [3]:
df_manifest = pd.read_csv(manifest_path)

print('df_manifest.shape:', df_manifest.shape)
print('df_manifest.columns:', list(df_manifest.columns))

# Protocolo: mostrar as primeiras 20 linhas
print('\nHEAD(20) do manifesto:')
display(df_manifest.head(20))



df_manifest.shape: (145, 24)
df_manifest.columns: ['id_arquivo', 'path_local', 'pasta_raiz', 'pasta_raiz_canonica', 'nome_arquivo', 'extensao', 'tamanho_bytes', 'data_modificacao', 'hash_sha256', 'dominio', 'forno', 'granularidade', 'origem_fisica', 'bucket_minio', 'path_minio', 'status_pipeline', 'fonte_manifesto', 'observacoes_dominio', 'observacoes_forno', 'observacoes_granularidade', 'observacoes_pipeline', 'fontes_manifesto', 'qtd_arquivos_reportada_manifesto', 'descricao_manifesto']

HEAD(20) do manifesto:


,id_arquivo,path_local,pasta_raiz,pasta_raiz_canonica,nome_arquivo,extensao,tamanho_bytes,data_modificacao,hash_sha256,dominio,forno,granularidade,origem_fisica,bucket_minio,path_minio,status_pipeline,fonte_manifesto,observacoes_dominio,observacoes_forno,observacoes_granularidade,observacoes_pipeline,fontes_manifesto,qtd_arquivos_reportada_manifesto,descricao_manifesto
0,0000b88858fe4a0371b224eb49a4f47f544c709322f38e...,dados/dados_iniciais/Consumo Fornos/2019_F1_Co...,Consumo Fornos,Consumo Fornos,2019_F1_Consumo.csv,.csv,42566405,2025-04-29T15:23:46+00:00,e2d6122f72312561518cd3dfbcfc425ca210fa1140a2e1...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/01/2019_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
1,e684247a48fd1fbc45f967124314b326cbc886b004fac0...,dados/dados_iniciais/Consumo Fornos/2020_F1_Co...,Consumo Fornos,Consumo Fornos,2020_F1_Consumo.csv,.csv,28785370,2025-04-29T15:25:02+00:00,e2f659eb79b9efb6d1a1aedf577b2f5ee9ce0b38cc8309...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2020_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
2,d2243389167f2f277659ebe70c82160fefd9ad15680e62...,dados/dados_iniciais/Consumo Fornos/2021_F1_Co...,Consumo Fornos,Consumo Fornos,2021_F1_Consumo.csv,.csv,29619879,2025-04-29T15:26:22+00:00,bca0695b76a186d6e603ddc6775d2379d4d094d2f66f89...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2021_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
3,df3919b433b24e293cb9a2fa9bb21e045dae7dffd169ff...,dados/dados_iniciais/Consumo Fornos/2022_F1_Co...,Consumo Fornos,Consumo Fornos,2022_F1_Consumo.csv,.csv,29118391,2025-04-29T15:27:40+00:00,7bd095e671301d332f6131d223a2343055f0f540e7e903...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2022_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
4,d31c7691b4c1d040fa135109d02b2fb0d0497587a1b309...,dados/dados_iniciais/Consumo Fornos/2023_F1_Co...,Consumo Fornos,Consumo Fornos,2023_F1_Consumo.csv,.csv,28111195,2025-04-29T15:28:52+00:00,600099b5cf736d738b126a60102b448129c63008357c31...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2023_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
5,23d9f0e4a454a7c4facf26bbbafb4be358f17b996a5924...,dados/dados_iniciais/Consumo Fornos/2024_F1_Co...,Consumo Fornos,Consumo Fornos,2024_F1_Consumo.csv,.csv,27148206,2025-04-29T15:30:04+00:00,488d5d081b8e32f72616373c97149e98f45c3ec71a83ba...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2024_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
6,d4635e66e027b7a960fc6fff149f5c3cb07b052c4a5409...,dados/dados_iniciais/Consumo Fornos/2025_F1_Co...,Consumo Fornos,Consumo

# ETAPA: Determinar coluna de path/arquivo no manifesto


In [4]:
# Heuristica: priorizar colunas completas de caminho (ex.: path_local) em vez de apenas nome do arquivo
candidates = []
colnames = [c.lower() for c in df_manifest.columns]

# Ordem de prioridade (mais completo primeiro)
preferred = [
    'path_local', 'path_minio', 'caminho_arquivo', 'caminho', 'path', 'filepath', 'file_path',
    'relative_path', 'arquivo', 'nome_arquivo', 'filename', 'file'
]

for p in preferred:
    for i, c in enumerate(df_manifest.columns):
        if c.lower() == p:
            candidates.append(c)

# fallback: procurar substrings
if not candidates:
    for i, c in enumerate(df_manifest.columns):
        cl = c.lower()
        if any(k in cl for k in ['caminho', 'path', 'arquivo', 'file']):
            candidates.append(c)

# Se ainda houver mais de uma, prioriza a que contem '/' (sugere caminho relativo)
if len(candidates) > 1:
    path_like = []
    for c in candidates:
        series_cat = df_manifest[c].astype(str).str.cat(sep='')
        if '/' in series_cat:
            path_like.append(c)
    if path_like:
        candidates = path_like

candidates = list(dict.fromkeys(candidates))

if len(candidates) >= 1:
    PATH_COL = candidates[0]
    if len(candidates) == 1:
        print('PATH_COL selecionada automaticamente:', PATH_COL)
    else:
        print('PATH_COL escolhida por prioridade:', PATH_COL)
        print('Outras candidatas:', candidates[1:])
else:
    print('Nao foi possivel selecionar unicamente a coluna de path.')
    print('Candidatas:', candidates)
    print('Defina PATH_COL manualmente na proxima linha e re-execute esta celula.')
    PATH_COL = None

PATH_COL



PATH_COL escolhida por prioridade: path_local
Outras candidatas: ['path_minio']


'path_local'

# ETAPA: Normalizar paths do manifesto


In [5]:
if PATH_COL is None:
    raise RuntimeError('PATH_COL nao definida. Selecione manualmente e reexecute.')


def normalize_path_value(v: object) -> str:
    if pd.isna(v):
        return ''
    s = str(v).strip().replace('\\\\', '/').replace('\\', '/')
    # remove prefixos redundantes
    while '//' in s:
        s = s.replace('//', '/')
    return s


def resolve_to_filesystem(p: str, raw_base: Path) -> Path:
    # Se ja for absoluto
    try:
        pp = Path(p)
    except Exception:
        return Path('')

    if pp.is_absolute():
        return pp

    # Se contem 'dados_iniciais', tenta ancorar a partir do raw_base
    if 'dados_iniciais' in p:
        # pega o trecho apos dados_iniciais/
        idx = p.find('dados_iniciais')
        tail = p[idx + len('dados_iniciais'):].lstrip('/').lstrip('\\')
        return raw_base / tail

    # default: relativo ao raw_base
    return raw_base / p


s_norm = df_manifest[PATH_COL].map(normalize_path_value)

resolved_paths = []
for p in tqdm(s_norm, desc='Resolvendo paths do manifesto'):
    rp = resolve_to_filesystem(p, dados_iniciais_dir)
    resolved_paths.append(rp)

# DataFrame novo (protocolo: head 20)
df_paths = pd.DataFrame({
    'manifest_row': np.arange(len(df_manifest)),
    'path_raw': df_manifest[PATH_COL],
    'path_norm': s_norm,
    'path_resolved': [str(p) for p in resolved_paths]
})

print('df_paths.shape:', df_paths.shape)
display(df_paths.head(20))



Resolvendo paths do manifesto: 100%|██████████| 145/145 [00:00<00:00, 79034.97it/s]

df_paths.shape: (145, 4)


,manifest_row,path_raw,path_norm,path_resolved
0,0,dados/dados_iniciais/Consumo Fornos/2019_F1_Co...,dados/dados_iniciais/Consumo Fornos/2019_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...
1,1,dados/dados_iniciais/Consumo Fornos/2020_F1_Co...,dados/dados_iniciais/Consumo Fornos/2020_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...
2,2,dados/dados_iniciais/Consumo Fornos/2021_F1_Co...,dados/dados_iniciais/Consumo Fornos/2021_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...
3,3,dados/dados_iniciais/Consumo Fornos/2022_F1_Co...,dados/dados_iniciais/Consumo Fornos/2022_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...
4,4,dados/dados_iniciais/Consumo Fornos/2023_F1_Co...,dados/dados_iniciais/Consumo Fornos/2023_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...
5,5,dados/dados_iniciais/Consumo Fornos/2024_F1_Co...,dados/dados_iniciais/Consumo Fornos/2024_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...
6,6,dados/dados_iniciais/Consumo Fornos/2025_F1_Co...,dados/dados_iniciais/Consumo Fornos/2025_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...
7,7,dados/dados_iniciais/Consumo Fornos/2018_F2_Co...,dados/dados_iniciais/Consumo Fornos/2018_F2_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...
8,8,dados/dados_iniciais/Consumo Fornos/2019_F2_Co...,dados/dados_iniciais/Consumo Fornos/2019_F2_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...
9,9,dados/dados_iniciais/Consumo Fornos/2020_F2_Co...,dados/dados_iniciais/Consumo Fornos/2020_F2_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...


# ETAPA: Validar manifesto -> filesystem (exists, size, mtime) e duplicidades


In [6]:
exists_list = []
size_list = []
mtime_list = []

for p_str in tqdm(df_paths['path_resolved'], desc='Checando existencia no filesystem'):
    p = Path(p_str)
    if p_str and p.exists() and p.is_file():
        exists_list.append(True)
        st = p.stat()
        size_list.append(st.st_size)
        mtime_list.append(st.st_mtime)
    else:
        exists_list.append(False)
        size_list.append(np.nan)
        mtime_list.append(np.nan)

# DataFrame novo (protocolo: head 20)
df_status = df_manifest.copy()
df_status['_path_col'] = PATH_COL
df_status['_path_norm'] = df_paths['path_norm']
df_status['_path_resolved'] = df_paths['path_resolved']
df_status['_exists'] = exists_list
df_status['_size_bytes'] = size_list
df_status['_mtime'] = mtime_list

print('df_status.shape:', df_status.shape)
display(df_status.head(20))

# Missing
missing = df_status[~df_status['_exists']].copy()
print('missing.shape:', missing.shape)
display(missing.head(20))

# Duplicidades no manifesto por path normalizado
# (mesmo arquivo referenciado em mais de uma linha)
dups = df_status[df_status['_path_norm'].astype(str).str.len() > 0].copy()
dups['_dup_count'] = dups.groupby('_path_norm')['_path_norm'].transform('count')
dups = dups[dups['_dup_count'] > 1].sort_values(['_dup_count','_path_norm'], ascending=[False, True])

print('dups.shape:', dups.shape)
display(dups.head(20))



Checando existencia no filesystem: 100%|██████████| 145/145 [00:00<00:00, 80616.92it/s]

df_status.shape: (145, 30)


,id_arquivo,path_local,pasta_raiz,pasta_raiz_canonica,nome_arquivo,extensao,tamanho_bytes,data_modificacao,hash_sha256,dominio,forno,granularidade,origem_fisica,bucket_minio,path_minio,status_pipeline,fonte_manifesto,observacoes_dominio,observacoes_forno,observacoes_granularidade,observacoes_pipeline,fontes_manifesto,qtd_arquivos_reportada_manifesto,descricao_manifesto,_path_col,_path_norm,_path_resolved,_exists,_size_bytes,_mtime
0,0000b88858fe4a0371b224eb49a4f47f544c709322f38e...,dados/dados_iniciais/Consumo Fornos/2019_F1_Co...,Consumo Fornos,Consumo Fornos,2019_F1_Consumo.csv,.csv,42566405,2025-04-29T15:23:46+00:00,e2d6122f72312561518cd3dfbcfc425ca210fa1140a2e1...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/01/2019_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM...",path_local,dados/dados_iniciais/Consumo Fornos/2019_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,True,42566405,1.745940e+09
1,e684247a48fd1fbc45f967124314b326cbc886b004fac0...,dados/dados_iniciais/Consumo Fornos/2020_F1_Co...,Consumo Fornos,Consumo Fornos,2020_F1_Consumo.csv,.csv,28785370,2025-04-29T15:25:02+00:00,e2f659eb79b9efb6d1a1aedf577b2f5ee9ce0b38cc8309...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2020_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM...",path_local,dados/dados_iniciais/Consumo Fornos/2020_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,True,28785370,1.745940e+09
2,d2243389167f2f277659ebe70c82160fefd9ad15680e62...,dados/dados_iniciais/Consumo Fornos/2021_F1_Co...,Consumo Fornos,Consumo Fornos,2021_F1_Consumo.csv,.csv,29619879,2025-04-29T15:26:22+00:00,bca0695b76a186d6e603ddc6775d2379d4d094d2f66f89...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2021_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM...",path_local,dados/dados_iniciais/Consumo Fornos/2021_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,True,29619879,1.745940e+09
3,df3919b433b24e293cb9a2fa9bb21e045dae7dffd169ff...,dados/dados_iniciais/Consumo Fornos/2022_F1_Co...,Consumo Fornos,Consumo Fornos,2022_F1_Consumo.csv,.csv,29118391,2025-04-29T15:27:40+00:00,7bd095e671301d332f6131d223a2343055f0f540e7e903...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2022_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM...",path_local,dados/dados_iniciais/Consumo Fornos/2022_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,True,29118391,1.745940e+09
4,d31c7691b4c1d040fa135109d02b2fb0d0497587a1b309...,dados/dados_iniciais/Consumo Fornos/2023_F1_Co...,Consumo Fornos,Consumo Fornos,2023_F1_Consumo.csv,.csv,28111195,2025-04-29T15:28:52+00:00,600099b5cf736d738b126a60102b448129c63008357c31...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2023_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM...",path_local,dados/dados_iniciais/Consumo Fornos/2023_F1_Co...,/home/wilson/Maringa/fase_1_diag

missing.shape: (0, 30)


,id_arquivo,path_local,pasta_raiz,pasta_raiz_canonica,nome_arquivo,extensao,tamanho_bytes,data_modificacao,hash_sha256,dominio,forno,granularidade,origem_fisica,bucket_minio,path_minio,status_pipeline,fonte_manifesto,observacoes_dominio,observacoes_forno,observacoes_granularidade,observacoes_pipeline,fontes_manifesto,qtd_arquivos_reportada_manifesto,descricao_manifesto,_path_col,_path_norm,_path_resolved,_exists,_size_bytes,_mtime


dups.shape: (0, 31)


,id_arquivo,path_local,pasta_raiz,pasta_raiz_canonica,nome_arquivo,extensao,tamanho_bytes,data_modificacao,hash_sha256,dominio,forno,granularidade,origem_fisica,bucket_minio,path_minio,status_pipeline,fonte_manifesto,observacoes_dominio,observacoes_forno,observacoes_granularidade,observacoes_pipeline,fontes_manifesto,qtd_arquivos_reportada_manifesto,descricao_manifesto,_path_col,_path_norm,_path_resolved,_exists,_size_bytes,_mtime,_dup_count


# ETAPA: Validar filesystem -> manifesto (arquivos orfaos no filesystem)


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm  # barra de texto

# 1) Normaliza manifesto para o mesmo universo do filesystem:
#    caminho relativo a dados_iniciais_dir (sempre com '/')
def to_rel_norm(p_str: str, base: Path) -> str:
    if not p_str:
        return ''
    p = Path(p_str)
    try:
        rel = p.resolve().relative_to(base.resolve())
        return rel.as_posix()
    except Exception:
        return p.resolve().as_posix()

# cria coluna com path relativo normalizado (a partir do path resolvido)
df_status['_fs_rel_norm'] = [
    to_rel_norm(p, dados_iniciais_dir)
    for p in tqdm(df_status['_path_resolved'], desc='Gerando rel_norm do manifesto')
]

print("df_status com _fs_rel_norm (head 20):")
display(df_status[['path_local','_path_resolved','_fs_rel_norm','_exists']].head(20))

manifest_set = set(df_status['_fs_rel_norm'].astype(str))

# 2) Varre filesystem sem filtrar extensoes (para nao subcontar)
fs_files = []
fs_norm = []

for p in tqdm(dados_iniciais_dir.rglob('*'), desc='Listando arquivos no filesystem'):
    if p.is_file():
        rel = p.relative_to(dados_iniciais_dir).as_posix()
        fs_files.append(str(p))
        fs_norm.append(rel)

# DataFrame novo relevante: head 20
df_fs = pd.DataFrame({
    'fs_path': fs_files,
    'fs_rel_norm': fs_norm
})

print('df_fs.shape:', df_fs.shape)
display(df_fs.head(20))

# 3) Orfaos: existem no filesystem mas nao aparecem no manifesto (em rel_norm)
orphan = df_fs[~df_fs['fs_rel_norm'].isin(manifest_set)].copy()
print('orphan.shape:', orphan.shape)
display(orphan.head(20))



Gerando rel_norm do manifesto: 100%|██████████| 145/145 [00:00<00:00, 27721.14it/s]

df_status com _fs_rel_norm (head 20):


,path_local,_path_resolved,_fs_rel_norm,_exists
0,dados/dados_iniciais/Consumo Fornos/2019_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2019_F1_Consumo.csv,True
1,dados/dados_iniciais/Consumo Fornos/2020_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2020_F1_Consumo.csv,True
2,dados/dados_iniciais/Consumo Fornos/2021_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2021_F1_Consumo.csv,True
3,dados/dados_iniciais/Consumo Fornos/2022_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2022_F1_Consumo.csv,True
4,dados/dados_iniciais/Consumo Fornos/2023_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2023_F1_Consumo.csv,True
5,dados/dados_iniciais/Consumo Fornos/2024_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2024_F1_Consumo.csv,True
6,dados/dados_iniciais/Consumo Fornos/2025_F1_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2025_F1_Consumo.csv,True
7,dados/dados_iniciais/Consumo Fornos/2018_F2_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2018_F2_Consumo.csv,True
8,dados/dados_iniciais/Consumo Fornos/2019_F2_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2019_F2_Consumo.csv,True
9,dados/dados_iniciais/Consumo Fornos/2020_F2_Co...,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2020_F2_Consumo.csv,True


Listando arquivos no filesystem: 154it [00:00, 91335.24it/s]

df_fs.shape: (145, 2)


,fs_path,fs_rel_norm
0,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Supervisorio Forno 4/F4_2024_2S.csv
1,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Supervisorio Forno 4/F4_2024_1S.csv
2,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Corridas/2020_F3_Corrida.csv
3,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Corridas/2019_F2_Corrida.csv
4,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Corridas/2018_F2_Corrida.csv
5,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Corridas/2019_F3_Corrida.csv
6,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Corridas/2023_F3_Corrida.csv
7,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Corridas/2024_F1_Corrida.csv
8,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Corridas/2023_F1_Corrida.csv
9,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Corridas/2025_F5_Corrida.csv


orphan.shape: (0, 2)


,fs_path,fs_rel_norm


# ETAPA: Persistir saidas e gerar relatorio R1


In [8]:
out_status = OUT_DIR / 'r1_manifest_status.csv'
out_missing = OUT_DIR / 'r1_missing_files.csv'
out_dups = OUT_DIR / 'r1_duplicate_manifest_entries.csv'
out_orphan = OUT_DIR / 'r1_orphan_files.csv'
out_summary = OUT_DIR / 'r1_summary.json'
out_report = OUT_DIR / 'r1_report.md'

# Persistencia
# Observacao: nao alterar dados de entrada, somente escrita em OUT_DIR

df_status.to_csv(out_status, index=False)
missing.to_csv(out_missing, index=False)
dups.to_csv(out_dups, index=False)
orphan.to_csv(out_orphan, index=False)

summary = {
    'run_ts': RUN_TS,
    'base_dir': str(BASE_DIR),
    'manifest_path': str(manifest_path),
    'dados_iniciais_dir': str(dados_iniciais_dir),
    'path_col': PATH_COL,
    'total_linhas_manifesto': int(df_status.shape[0]),
    'arquivos_unicos_manifesto': int(df_status['_path_norm'].nunique()),
    'encontrados_no_fs': int(df_status['_exists'].sum()),
    'ausentes_no_fs': int((~df_status['_exists']).sum()),
    'duplicados_no_manifesto_linhas': int(dups.shape[0]),
    'total_arquivos_fs': int(df_fs.shape[0]),
    'orfaos_no_fs': int(orphan.shape[0])
}

out_summary.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

# Relatorio objetivo em Markdown
report_lines = []
report_lines.append('# Retomada R1 - Relatorio de Validacao')
report_lines.append('')
report_lines.append(f'run_ts: {RUN_TS}')
report_lines.append(f'manifest_path: {manifest_path}')
report_lines.append(f'dados_iniciais_dir: {dados_iniciais_dir}')
report_lines.append(f'path_col: {PATH_COL}')
report_lines.append('')
report_lines.append('## Sumario')
for k, v in summary.items():
    if k in ['run_ts','base_dir','manifest_path','dados_iniciais_dir','path_col']:
        continue
    report_lines.append(f'- {k}: {v}')

report_lines.append('')
report_lines.append('## Artefatos gerados')
report_lines.append(f'- {out_status.name}')
report_lines.append(f'- {out_missing.name}')
report_lines.append(f'- {out_dups.name}')
report_lines.append(f'- {out_orphan.name}')
report_lines.append(f'- {out_summary.name}')
report_lines.append('')
report_lines.append('## Amostras (top 20)')
report_lines.append('')
report_lines.append('### Missing files (top 20)')
if missing.shape[0] == 0:
    report_lines.append('Nenhum.')
else:
    for s in missing['_path_norm'].astype(str).head(20).tolist():
        report_lines.append(f'- {s}')

report_lines.append('')
report_lines.append('### Orphan files (top 20)')
if orphan.shape[0] == 0:
    report_lines.append('Nenhum.')
else:
    for s in orphan['fs_rel_norm'].astype(str).head(20).tolist():
        report_lines.append(f'- {s}')

out_report.write_text('\n'.join(report_lines), encoding='utf-8')

print('Arquivos gerados:')
for p in [out_status, out_missing, out_dups, out_orphan, out_summary, out_report]:
    print(' -', p)

print('\nResumo:')
print(json.dumps(summary, ensure_ascii=False, indent=2))



Arquivos gerados:
 - /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R1/r1_manifest_status.csv
 - /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R1/r1_missing_files.csv
 - /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R1/r1_duplicate_manifest_entries.csv
 - /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R1/r1_orphan_files.csv
 - /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R1/r1_summary.json
 - /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R1/r1_report.md

Resumo:
{
  "run_ts": "2026-01-05_162203",
  "base_dir": "/home/wilson/Maringa/fase_1_diagnostico",
  "manifest_path": "/home/wilson/Maringa/fase_1_diagnostico/dados/manifestos_final/manifest_unificado.csv",
  "dados_iniciais_dir": "/home/wilson/Maringa/fase_1_diagnostico/dados/dados_iniciais",
  "path_col": "path_local",
  "total_linhas_manifesto": 145,
  "arquivos_unicos_manifesto": 145,
  "encontrados_no_fs": 145,
  "ausentes_no_fs": 0,
  "duplicados_